# 05 — Advanced Patterns: MCP & Agent-to-Agent

**Session: Month 6, Session 1**

## Learning Objectives
1. Understand the Model Context Protocol (MCP) and why it matters
2. Discover and invoke tools via the MCP server
3. Implement the A2A message protocol between agents
4. Design a multi-agent workflow using the hierarchical orchestrator

## Why MCP?
Without MCP, every AI host (Claude Desktop, Cursor, your app) needs custom
integration code for every tool. With MCP, tools are discoverable — any
MCP-compatible client can discover and use them automatically.

```
Before MCP:                After MCP:
App A ──custom──► Tool     App A ──MCP──► Tool Registry
App B ──custom──► Tool     App B ──MCP──►     (all tools)
App C ──custom──► Tool     App C ──MCP──►
```

In [ ]:
import sys, asyncio, json
sys.path.insert(0, '../backend')

from app.core.a2a_protocol import MessageBus, A2AMessage, AgentCapability, MessageType, get_message_bus
from app.core.orchestrator import AgentOrchestrator, QueryAgent, TransactionAgent

## Part 1: MCP Tool Discovery

In [ ]:
from app.tools.registry import ALL_TOOLS

print(f'Total tools registered: {len(ALL_TOOLS)}')
print()

# Simulate MCP tool manifest
READ_TOOLS = {'lookup_order', 'get_customer_info', 'find_customer_by_email',
              'get_customer_order_history', 'search_products', 'search_documents'}
WRITE_TOOLS = {t.name for t in ALL_TOOLS if t.name not in READ_TOOLS}

print('READ tools (readonly agent access):')
for t in ALL_TOOLS:
    if t.name in READ_TOOLS:
        desc = t.description.split('\n')[0][:55]
        print(f'  📖 {t.name:<35} {desc}')

print('\nWRITE tools (transaction agent access):')
for t in ALL_TOOLS:
    if t.name in WRITE_TOOLS:
        desc = t.description.split('\n')[0][:55]
        print(f'  ✏️  {t.name:<35} {desc}')

## Part 2: Agent-to-Agent (A2A) Protocol

In [ ]:
# Simulate A2A message passing between Supervisor and QueryAgent

bus = MessageBus()

# Register a simulated QueryAgent handler
async def query_agent_handler(message: A2AMessage):
    print(f'  QueryAgent received: {message.payload}')
    # Simulate looking up order
    await asyncio.sleep(0.1)
    return A2AMessage.response(
        original=message,
        payload={
            'result': {
                'order_id': 'ORD-10001',
                'status': 'shipped',
                'tracking': 'TRK-ABC123'
            }
        },
        sender='query-agent',
    )

bus.register(
    AgentCapability(
        agent_id='query-agent',
        name='Query Specialist',
        description='Handles read-only lookups',
        supported_intents=['lookup', 'search', 'info'],
        tool_access='readonly',
    ),
    query_agent_handler,
)

print('A2A Demo: Supervisor → QueryAgent')
print('=' * 50)

# Supervisor sends a request
request = A2AMessage.request(
    sender='supervisor',
    recipient='query-agent',
    conversation_id='conv-demo-001',
    payload={'user_message': 'What is the status of order ORD-10001?', 'tool': 'lookup_order'},
)
print(f'\nSupervisor sends: {json.dumps(request.payload, indent=2)}')

response = await bus.publish(request)
print(f'\nQueryAgent responds: {json.dumps(response.payload, indent=2)}')
print(f'\nMessage log ({len(bus.get_message_log("conv-demo-001"))} messages):')
for msg in bus.get_message_log('conv-demo-001'):
    print(f'  [{msg["type"]}] {msg["sender"]} → {msg["recipient"]} | conv={msg["conversation_id"]}')

## Part 3: Hierarchical Orchestrator Routing

In [ ]:
# Test the routing logic (without real LLM — inspect the routing logic)
from app.core.orchestrator import AgentOrchestrator, READONLY_TOOLS, WRITE_TOOLS

print('Supervisor Routing Logic:')
print('=' * 60)

test_requests = [
    'What is the status of my order ORD-10001?',
    'Cancel my order ORD-10002 please',
    'I need a refund for order ORD-10003',
    'What is your return policy?',
    'Apply discount code SAVE20 to my order ORD-10004',
    'Speed up my delivery for order ORD-10005',
]

# Simple heuristic routing (keyword-based, as the LLM is not running here)
def simple_route(msg):
    msg_lower = msg.lower()
    write_keywords = {'cancel', 'refund', 'apply', 'change', 'update', 'speed up', 'expedite', 'discount'}
    if any(kw in msg_lower for kw in write_keywords):
        return 'transaction'
    return 'query'

print(f'{"Request":<55} {"Routed To"}')
print('-' * 70)
for req in test_requests:
    route = simple_route(req)
    icon = '💰' if route == 'transaction' else '🔍'
    print(f'{req[:55]:<55} {icon} {route}')

print(f'\nQuery Agent has {len(READONLY_TOOLS)} tools (read-only)')
print(f'Transaction Agent has {len(WRITE_TOOLS) + len(READONLY_TOOLS)} tools (all)')

## Lab Exercise: Add a Compliance Agent

Add a third specialist agent: `ComplianceAgent`

This agent runs **after** every transaction and checks:
1. Was the refund amount proportional to the order total?
2. Was there a valid approval for transactions over $500?
3. Was the audit log entry created?

If any check fails, it creates a compliance incident.

**Implementation steps:**
1. Create `ComplianceAgent` class in `orchestrator.py`
2. Register it with the A2A `MessageBus`
3. Have `AgentOrchestrator.process()` send a broadcast after each transaction
4. Add a `/api/compliance/incidents` endpoint
5. Write tests for the compliance checks

This is the **monitoring pattern** used in regulated industries (banking, healthcare).